I will be backtesting some portfolio allocation strategies using the return series of 30 industries from 1926 till 2018.

Importing all the required modules:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats
from scipy.optimize import minimize
plt.style.use('seaborn-v0_8')

The data I will be working with contains the monthly returns of 30 different industries from the second half of 1926 till the end of 2018: 

In [2]:
# 'ind' variable contains the monthly returns
ind_r = pd.read_csv("ind30_monthly_returns.csv", header=0, index_col=0)/100
ind_r.index = pd.to_datetime(ind_r.index, format="%Y%m").to_period('M')
ind_r.columns = ind_r.columns.str.strip()
ind_r

,Food,Beer,Smoke,Games,Books,Hshld,Clths,Hlth,Chems,Txtls,...,Telcm,Servs,BusEq,Paper,Trans,Whlsl,Rtail,Meals,Fin,Other
1926-07,0.0056,-0.0519,0.0129,0.0293,0.1097,-0.0048,0.0808,0.0177,0.0814,0.0039,...,0.0083,0.0922,0.0206,0.0770,0.0193,-0.2379,0.0007,0.0187,0.0037,0.0520
1926-08,0.0259,0.2703,0.0650,0.0055,0.1001,-0.0358,-0.0251,0.0425,0.0550,0.0814,...,0.0217,0.0202,0.0439,-0.0238,0.0488,0.0539,-0.0075,-0.0013,0.0446,0.0676
1926-09,0.0116,0.0402,0.0126,0.0658,-0.0099,0.0073,-0.0051,0.0069,0.0533,0.0231,...,0.0241,0.0225,0.0019,-0.0554,0.0005,-0.0787,0.0025,-0.0056,-0.0123,-0.0386
1926-10,-0.0306,-0.0331,0.0106,-0.0476,0.0947,-0.0468,0.0012,-0.0057,-0.0476,0.0100,...,-0.0011,-0.0200,-0.0109,-0.0508,-0.0264,-0.1538,-0.0220,-0.0411,-0.0516,-0.0849
1926-11,0.0635,0.0729,0.0455,0.0166,-0.0580,-0.0054,0.0187,0.0542,0.0520,0.0311,...,0.0163,0.0377,0.0364,0.0384,0.0160,0.0467,0.0652,0.0433,0.0224,0.0400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-08,-0.0038,-0.0186,-0.0543,0.0289,-0.0447,0.0254,0.0526,0.0446,0.0001,0.0380,...,0.0295,0.0524,0.0993,-0.0034,0.0291,0.0366,0.0911,0.0364,0.0245,0.0299
2018-09,-0.0032,0.0019,0.0537,-0.0009,-0.0221,0.0107,0.0130,0.0199,-0.0287,-0.0638,...,0.0174,-0.0037,-0.0033,-0.0030,0.0105,-0.0148,0.0061,0.0251,-0.0193,0.0116
2018-10,0.0102,-0.0157,0.0790,-0.1596,-0.0666,-0.0051,-0.1014,-0.0884,-0.1250,-0.2579,...,-0.0050,-0.0920,-0.0806,-0.0982,-0.0975,-0.0788,-0.1021,-0.0171,-0.0545,-0.0599
2018-11,0.0272,0.0579,-0.0843,-0.0065,0.0325,0.0644,-0.0099,0.0632,0.0496,0.0292,...,0.0254,0.0129,-0.0505,0.0822,0.0617,0.0318,0.0159,0.0616,0.0289,0.0348


I will also create variables that contain the size of each industry and the number of firms in each industry as this will be needed when I later create a cap-weighted market index:

In [3]:
ind_size = pd.read_csv("ind30_m_size.csv", header=0, index_col=0)
ind_size.index = pd.to_datetime(ind_size.index, format="%Y%m").to_period('M')
ind_size.columns = ind_size.columns.str.strip()
ind_size

,Food,Beer,Smoke,Games,Books,Hshld,Clths,Hlth,Chems,Txtls,...,Telcm,Servs,BusEq,Paper,Trans,Whlsl,Rtail,Meals,Fin,Other
1926-07,35.98,7.12,59.72,26.41,12.02,22.27,18.36,25.52,57.59,6.18,...,350.36,13.60,56.70,35.35,66.91,1.19,46.65,10.82,18.83,24.25
1926-08,36.10,6.75,60.47,27.17,13.33,22.13,19.83,25.80,62.13,6.20,...,353.27,14.75,57.74,37.86,67.99,0.90,46.57,11.00,18.88,25.51
1926-09,37.00,8.58,64.03,27.30,14.67,21.18,19.29,26.73,65.53,6.71,...,360.96,15.05,59.61,36.82,71.02,0.95,46.11,10.94,19.67,27.21
1926-10,37.14,8.92,64.42,28.76,14.42,21.23,19.03,26.87,68.47,6.82,...,364.16,15.30,59.52,34.77,70.83,0.88,46.15,10.80,19.36,26.16
1926-11,35.88,8.62,65.08,27.38,15.79,20.14,19.03,26.54,65.06,6.84,...,363.74,14.89,58.74,32.80,68.75,0.74,45.03,10.33,18.35,23.94
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-08,11697.98,20738.70,82083.79,5614.41,1666.00,11188.09,9703.80,4891.61,7750.06,2307.51,...,17316.92,10961.79,11431.81,8167.26,10034.30,3392.45,15964.58,6806.11,7632.93,11015.79
2018-09,11624.31,20332.85,77629.48,5773.17,1587.92,11463.79,10199.85,5115.56,7726.66,2394.61,...,17816.83,11522.53,12631.96,8103.55,10292.17,3511.81,17387.93,7135.07,7834.57,11334.95
2018-10,11614.93,20256.16,80703.59,5816.00,1549.21,11579.42,10661.35,5228.28,6901.05,2241.00,...,18118.32,11540.36,12666.71,8064.22,10395.22,3476.28,17611.78,7312.36,7727.88,11453.54
2018-11,11721.21,21743.93,87079.79,4962.71,1444.47,11460.11,9579.97,4777.85,6113.67,1663.11,...,17934.51,10505.66,11635.87,7379.17,9513.99,3198.05,15917.02,7298.54,7217.76,10931.44


In [4]:
ind_n = pd.read_csv("ind30_m_number_of_firms.csv", header=0, index_col=0)
ind_n.index = pd.to_datetime(ind_n.index, format="%Y%m").to_period('M')
ind_n.columns = ind_n.columns.str.strip()
ind_n

,Food,Beer,Smoke,Games,Books,Hshld,Clths,Hlth,Chems,Txtls,...,Telcm,Servs,BusEq,Paper,Trans,Whlsl,Rtail,Meals,Fin,Other
1926-07,43,3,16,7,2,8,12,7,17,13,...,5,3,7,6,74,2,33,6,12,4
1926-08,43,3,16,7,2,8,12,7,17,13,...,5,3,7,6,74,2,33,6,12,4
1926-09,43,3,16,7,2,8,12,7,17,13,...,5,3,7,6,74,2,33,6,12,4
1926-10,43,3,16,7,2,8,12,7,17,13,...,5,3,7,6,74,2,33,6,12,4
1926-11,43,3,16,7,2,8,12,7,17,13,...,5,3,7,6,74,2,33,6,12,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-08,56,12,3,64,21,39,30,634,68,8,...,69,440,284,37,70,97,143,63,646,130
2018-09,56,12,3,64,21,39,30,632,68,8,...,69,440,282,37,70,97,143,62,643,130
2018-10,55,12,3,63,21,39,29,629,67,8,...,69,437,280,37,70,95,142,62,638,130
2018-11,55,11,3,62,21,39,29,626,66,8,...,69,434,280,36,69,95,141,61,635,128


Before creating and backtesting strategies, I first need a benchmark to compare the performance of other strategies to. A good benchmark is the cap-weighted market index:

In [5]:
ind_mktcap = ind_n * ind_size
total_mktcap = ind_mktcap.sum(axis=1)
ind_capweight = ind_mktcap.divide(total_mktcap, axis="rows")
total_market_return = (ind_capweight * ind_r).sum(axis="columns")
total_market_return

1926-07    0.031375
1926-08    0.028957
1926-09    0.005566
1926-10   -0.028504
1926-11    0.028039
             ...   
2018-08    0.036951
2018-09    0.002108
2018-10   -0.074292
2018-11    0.019003
2018-12   -0.092911
Freq: M, Length: 1110, dtype: float64

I will be implementing the different strategies from 1935 onwards, so I will create a new variable that contains the returns of the cap-weighted market index only from 1935 onwards:

In [6]:
market_rets = total_market_return['1935':]
market_rets

1935-01   -0.034309
1935-02   -0.019602
1935-03   -0.035934
1935-04    0.089837
1935-05    0.035027
             ...   
2018-08    0.036951
2018-09    0.002108
2018-10   -0.074292
2018-11    0.019003
2018-12   -0.092911
Freq: M, Length: 1008, dtype: float64

We want to understand this return series(and also other series when we eventually create other strategies), specifically the different useful statistics that can be calculated like the annualized return and volatility. Below, I will create a function called 'summary_stats', that returns useful statistics of a return series when you input the returns and the risk free rate:

In [7]:
def annualize_rets(r, periods_per_year):
    """
    Annualizes a set of returns
    We should infer the periods per year
    but that is currently left as an exercise
    to the reader :-)
    """
    compounded_growth = (1+r).prod()
    n_periods = r.shape[0]
    return compounded_growth**(periods_per_year/n_periods)-1

def annualize_vol(r, periods_per_year):
    """
    Annualizes the vol of a set of returns
    We should infer the periods per year
    but that is currently left as an exercise
    to the reader :-)
    """
    return r.std()*(periods_per_year**0.5)

def sharpe_ratio(r, riskfree_rate, periods_per_year):
    """
    Computes the annualized sharpe ratio of a set of returns
    """
    # convert the annual riskfree rate to per period
    rf_per_period = (1+riskfree_rate)**(1/periods_per_year)-1
    excess_ret = r - rf_per_period
    ann_ex_ret = annualize_rets(excess_ret, periods_per_year)
    ann_vol = annualize_vol(r, periods_per_year)
    return ann_ex_ret/ann_vol

def drawdown(return_series: pd.Series):
    """Takes a time series of asset returns.
       returns a DataFrame with columns for
       the wealth index, 
       the previous peaks, and 
       the percentage drawdown
    """
    wealth_index = 1000*(1+return_series).cumprod()
    previous_peaks = wealth_index.cummax()
    drawdowns = (wealth_index - previous_peaks)/previous_peaks
    return pd.DataFrame({"Wealth": wealth_index, 
                         "Previous Peak": previous_peaks, 
                         "Drawdown": drawdowns})

def skewness(r):
    """
    Alternative to scipy.stats.skew()
    Computes the skewness of the supplied Series or DataFrame
    Returns a float or a Series
    """
    demeaned_r = r - r.mean()
    # use the population standard deviation, so set dof=0
    sigma_r = r.std(ddof=0)
    exp = (demeaned_r**3).mean()
    return exp/sigma_r**3

def kurtosis(r):
    """
    Alternative to scipy.stats.kurtosis()
    Computes the kurtosis of the supplied Series or DataFrame
    Returns a float or a Series
    """
    demeaned_r = r - r.mean()
    # use the population standard deviation, so set dof=0
    sigma_r = r.std(ddof=0)
    exp = (demeaned_r**4).mean()
    return exp/sigma_r**4

from scipy.stats import norm
def var_gaussian(r, level=5, modified=False):
    """
    Returns the Parametric Gauusian VaR of a Series or DataFrame
    If "modified" is True, then the modified VaR is returned,
    using the Cornish-Fisher modification
    """
    # compute the Z score assuming it was Gaussian
    z = norm.ppf(level/100)
    if modified:
        # modify the Z score based on observed skewness and kurtosis
        s = skewness(r)
        k = kurtosis(r)
        z = (z +
                (z**2 - 1)*s/6 +
                (z**3 -3*z)*(k-3)/24 -
                (2*z**3 - 5*z)*(s**2)/36
            )
    return -(r.mean() + z*r.std(ddof=0))

def var_historic(r, level=5):
    """
    Returns the historic Value at Risk at a specified level
    i.e. returns the number such that "level" percent of the returns
    fall below that number, and the (100-level) percent are above
    """
    if isinstance(r, pd.DataFrame):
        return r.aggregate(var_historic, level=level)
    elif isinstance(r, pd.Series):
        return -np.percentile(r, level)
    else:
        raise TypeError("Expected r to be a Series or DataFrame")

def cvar_historic(r, level=5):
    """
    Computes the Conditional VaR of Series or DataFrame
    """
    if isinstance(r, pd.Series):
        is_beyond = r <= -var_historic(r, level=level)
        return -r[is_beyond].mean()
    elif isinstance(r, pd.DataFrame):
        return r.aggregate(cvar_historic, level=level)
    else:
        raise TypeError("Expected r to be a Series or DataFrame")
        
def summary_stats(r, riskfree_rate=0.03):
    """
    Return a DataFrame that contains aggregated summary stats for the returns in the columns of r
    """
    ann_r = r.aggregate(annualize_rets, periods_per_year=12)
    ann_vol = r.aggregate(annualize_vol, periods_per_year=12)
    ann_sr = r.aggregate(sharpe_ratio, riskfree_rate=riskfree_rate, periods_per_year=12)
    dd = r.aggregate(lambda r: drawdown(r).Drawdown.min())
    skew = r.aggregate(skewness)
    kurt = r.aggregate(kurtosis)
    cf_var5 = r.aggregate(var_gaussian, modified=True)
    hist_cvar5 = r.aggregate(cvar_historic)
    return pd.DataFrame({
        "Annualized Return": ann_r,
        "Annualized Vol": ann_vol,
        "Sharpe Ratio": ann_sr,
        "Skewness": skew,
        "Kurtosis": kurt,
        "Cornish-Fisher VaR (5%)": cf_var5,
        "Historic CVaR (5%)": hist_cvar5,
        "Max Drawdown": dd
    })

I will now calculate the summary statistics for the return series of the cap-weighted market index. I will assume that the risk free rate is constant at 3%.

In [8]:
#Converting to DataFrame so summary_stats works perfectly
market_df = pd.DataFrame(market_rets, columns=["MarketCap_Weighted"])

summary_stats(market_df, riskfree_rate=0.03)

,Annualized Return,Annualized Vol,Sharpe Ratio,Skewness,Kurtosis,Cornish-Fisher VaR (5%),Historic CVaR (5%),Max Drawdown
MarketCap_Weighted,0.109958,0.155409,0.500578,-0.530726,6.174386,0.067662,0.09728,-0.499943


Now that I have computed the summary statistics for the cap-weighted index, I will move on to creating some portfolio allocation strategies, backtesting them from 1935 onwards, and computing their summary statistics. First, I will create two functions: 'portfolio_return' and 'portfolio_vol', which calculate the portfolio return and portfolio volatility respectively, given allocation weights, returns(for portfolio return), and covariance matrix(for portfolio volatility).

In [9]:
def portfolio_return(weights, returns):
    """
    Computes the return on a portfolio from constituent returns and weights.
    Weights are a numpy array or Nx1 matrix and returns are a numpy array or Nx1 matrix
    """
    return weights.T @ returns


def portfolio_vol(weights, covmat):
    """
    Computes the vol of a portfolio from a covariance matrix and constituent weights
    weights are a numpy array or N x 1 maxtrix and covmat is an N x N matrix
    """
    return (weights.T @ covmat @ weights)**0.5

The first strategy I will be developing is the 'Rolling Max Sharpe Ratio' strategy. This strategy calculates the fixed weights that would need to be allocated to the different industries to obtain the maximum sharpe ratio over a particular time period in the past, and applies those weights for the next month. This will be repeated at the beginning of every month. For example, if the time period chosen is the last 5 years, then this strategy will calculate the weights that need to be allocated to the different industries to achieve the maximum sharpe ratio across the last 5 years; these weights will then be applied for the current month, then at the beginning of each new month, this process will be repeated. To develop this strategy, I will first define the function 'msr', which - when inputted the risk free rate, expected return(historical return in this case), and covariance matrix - will return the weights of the portfolio that gives the maximum sharpe ratio:

In [10]:
def msr(riskfree_rate, er, cov):
    """
    Returns the weights of the portfolio that gives you the maximum sharpe ratio
    given the riskfree rate and expected returns and a covariance matrix
    """
    n = er.shape[0]
    init_guess = np.repeat(1/n, n)
    bounds = ((0.0, 1.0),) * n # an N-tuple of 2-tuples!
    # construct the constraints
    weights_sum_to_1 = {'type': 'eq',
                        'fun': lambda weights: np.sum(weights) - 1
    }
    def neg_sharpe(weights, riskfree_rate, er, cov):
        """
        Returns the negative of the sharpe ratio
        of the given portfolio
        """
        r = portfolio_return(weights, er)
        vol = portfolio_vol(weights, cov)
        return -(r - riskfree_rate)/vol
    
    weights = minimize(neg_sharpe, init_guess,
                       args=(riskfree_rate, er, cov), method='SLSQP',
                       options={'disp': False},
                       constraints=(weights_sum_to_1,),
                       bounds=bounds)
    return weights.x

Now, I will develop the function for the Rolling MSR strategy backtest. This function - when inputted the return series, length of time period in the past to calculate the weights, and the risk free rate - will return the monthly returns of this particular strategy. 

In [30]:
#ROLLING MSR BACKTEST (STARTS 1935)

def rolling_msr_backtest(rets, window_months, riskfree_rate=0.03):
    """
    Rolling Maximum Sharpe Ratio strategy.
    Uses previous 60 months to optimize, applies weights to next month.
    """
    portfolio_returns = []
    dates_used = []
    
    for t in range(window_months, len(rets)):
        
        train = rets.iloc[t - window_months : t]
        
        er = train.mean() * 12
        cov = train.cov() * 12
        
        # Get msr weights using msr function
        w = msr(riskfree_rate, er, cov)
        
        # Apply to the NEXT month (out-of-sample)
        next_ret = rets.iloc[t] @ w
        
        portfolio_returns.append(next_ret)
        dates_used.append(rets.index[t])
    
    return pd.Series(portfolio_returns, index=dates_used, name="Rolling_MSR")

Now that the function is defined, I can run the strategy using different values of the 'window_months'(length of time period used in calculating the max sharpe ratio weights).

In [12]:
# Run the rolling MSR strategy on the full dataset
rolling_msr_rets_60 = rolling_msr_backtest(ind_r, window_months=60)
rolling_msr_rets_50 = rolling_msr_backtest(ind_r, window_months=50)
rolling_msr_rets_40 = rolling_msr_backtest(ind_r, window_months=40)
rolling_msr_rets_30 = rolling_msr_backtest(ind_r, window_months=30)
rolling_msr_rets_20 = rolling_msr_backtest(ind_r, window_months=20)
rolling_msr_rets_10 = rolling_msr_backtest(ind_r, window_months=10)

# Slice everything to start from 1935 (strategy starts investing Jan 1935)
start_date = '1935'

rolling_msr_rets_60m = rolling_msr_rets_60[start_date:]
rolling_msr_rets_50m = rolling_msr_rets_50[start_date:]
rolling_msr_rets_40m = rolling_msr_rets_40[start_date:]
rolling_msr_rets_30m = rolling_msr_rets_30[start_date:]
rolling_msr_rets_20m = rolling_msr_rets_20[start_date:]
rolling_msr_rets_10m = rolling_msr_rets_10[start_date:]

comparison = pd.DataFrame({
    "Rolling MSR Strategy (60 Months)": rolling_msr_rets_60m,
    "Rolling MSR Strategy (50 Months)": rolling_msr_rets_50m,
    "Rolling MSR Strategy (40 Months)": rolling_msr_rets_40m,
    "Rolling MSR Strategy (30 Months)": rolling_msr_rets_30m,
    "Rolling MSR Strategy (20 Months)": rolling_msr_rets_20m,
    "Rolling MSR Strategy (10 Months)": rolling_msr_rets_10m,
    "Cap-weighted Market Index": market_rets
    
})

print(summary_stats(comparison))

                                  Annualized Return  Annualized Vol  \
Rolling MSR Strategy (60 Months)           0.122959        0.168710   
Rolling MSR Strategy (50 Months)           0.109089        0.169534   
Rolling MSR Strategy (40 Months)           0.103890        0.172895   
Rolling MSR Strategy (30 Months)           0.114894        0.177555   
Rolling MSR Strategy (20 Months)           0.138258        0.190336   
Rolling MSR Strategy (10 Months)           0.137150        0.187077   
Cap-weighted Market Index                  0.109958        0.155409   

                                  Sharpe Ratio  Skewness   Kurtosis  \
Rolling MSR Strategy (60 Months)      0.536096 -0.045758   5.473518   
Rolling MSR Strategy (50 Months)      0.453849 -0.299676   4.653936   
Rolling MSR Strategy (40 Months)      0.415745 -0.434652   5.591496   
Rolling MSR Strategy (30 Months)      0.465150 -0.355236   5.725791   
Rolling MSR Strategy (20 Months)      0.553369 -0.076401   9.307823   
Rolli

There does not seem to be a trend in the sharpe ratios as the length of the time window changes. The 60, 20, and 10 month window strategies had higher sharpe ratios than the cap-weighted market index, whereas the 50, 40, and 30 month window strategies hard lower sharpe ratios than the cap-weighted market index. The highest sharpe ratio are the 20 and 10 month window strategies with sharpe ratios between 0.55 and 0.56. The lowest is the 40 month window strategy with a sharpe ratio of 0.42. Compared to the market index, on average, this strategy has a higher max drawdown. The higher Cornish Fisher VaR and Historic Conditional VaR values also indicate higher tail risk compared to the market index.

I will now implement a strategy that is an extension to the Rolling MSR strategy. This strategy will allocate between a risk-free asset with a 3% annual return, and the Rolling MSR strategy(Risky Asset). The allocation to the risky asset will increase when previous month's return was positive, and will decrease when previous month's return was negative. The extent to which the allocation to the risky asset will increase or decrease depends on the size of the previous month's return in absolute terms. The initial allocation to the risky asset will be 100%.

In [31]:
#DYNAMIC ALLOCATION ROLLING MSR

def rolling_msr_dynamic_allocation(rets, 
                                   window_months=60, 
                                   riskfree_rate=0.03, 
                                   sensitivity=8.0,      # higher = more aggressive reaction to last month
                                   max_risky_weight=1.0, 
                                   start_date='1935'):
    """
    Rolling MSR + Dynamic allocation between MSR portfolio and risk-free asset.
    Allocation to risky asset increases when previous month's total return was higher.
    """
    portfolio_returns = []
    risky_weights_history = []
    dates_used = []
    
    # Initial allocation (start with 100% in risky)
    current_risky_weight = 1.0
    
    for t in range(window_months, len(rets)):
        # 1. Training window for optimization
        train = rets.iloc[t - window_months : t]
        er = train.mean() * 12
        cov = train.cov() * 12
        
        # 2. Compute current MSR portfolio weights
        w_msr = msr(riskfree_rate, er, cov)
        
        # 3. Compute what the TOTAL portfolio return was LAST month
        if t > window_months:  # after the first month
            last_month_risky_ret = rets.iloc[t-1] @ w_msr   # use the weights from previous optimization
            last_month_total_ret = (current_risky_weight * last_month_risky_ret) + \
                                   ((1 - current_risky_weight) * (riskfree_rate / 12))
        else:
            last_month_total_ret = 0.0
        
        # 4. Update risky allocation based on last month's total return
        current_risky_weight = current_risky_weight + sensitivity * last_month_total_ret
        current_risky_weight = np.clip(current_risky_weight, 0.0, max_risky_weight)
        
        # 5. Compute this month's portfolio return
        this_month_risky_ret = rets.iloc[t] @ w_msr
        this_month_total_ret = (current_risky_weight * this_month_risky_ret) + \
                               ((1 - current_risky_weight) * (riskfree_rate / 12))
        
        portfolio_returns.append(this_month_total_ret)
        risky_weights_history.append(current_risky_weight)
        dates_used.append(rets.index[t])
    
    result = pd.Series(portfolio_returns, index=dates_used, name="Dynamic_Rolling_MSR")
    weights_series = pd.Series(risky_weights_history, index=dates_used, name="Risky_Weight")
    
    return result

In [32]:
#RUNNING THE STRATEGY

# Run the dynamic strategy
dynamic_rets_60 = rolling_msr_dynamic_allocation(rets=ind_r, window_months=60)
dynamic_rets_50 = rolling_msr_dynamic_allocation(rets=ind_r, window_months=50)
dynamic_rets_40 = rolling_msr_dynamic_allocation(rets=ind_r, window_months=40)
dynamic_rets_30 = rolling_msr_dynamic_allocation(rets=ind_r, window_months=30)
dynamic_rets_20 = rolling_msr_dynamic_allocation(rets=ind_r, window_months=20)
dynamic_rets_10 = rolling_msr_dynamic_allocation(rets=ind_r, window_months=10)
                
# Align from 1935 onwards
start_date = '1935'
dynamic_rets_60m = dynamic_rets_60[start_date:]
dynamic_rets_50m = dynamic_rets_50[start_date:]
dynamic_rets_40m = dynamic_rets_40[start_date:]
dynamic_rets_30m = dynamic_rets_30[start_date:]
dynamic_rets_20m = dynamic_rets_20[start_date:]
dynamic_rets_10m = dynamic_rets_10[start_date:]

# Comparison
comparison = pd.DataFrame({
    "Dynamic Rolling MSR(60 Months)": dynamic_rets_60m,
    "Dynamic Rolling MSR(50 Months)": dynamic_rets_50m,
    "Dynamic Rolling MSR(40 Months)": dynamic_rets_40m,
    "Dynamic Rolling MSR(30 Months)": dynamic_rets_30m,
    "Dynamic Rolling MSR(20 Months)": dynamic_rets_20m,
    "Dynamic Rolling MSR(10 Months)": dynamic_rets_10m,
    "Cap-weighted Market Index": market_rets
})

print(f"=== PERFORMANCE FROM {start_date} ONWARDS ===")
print(summary_stats(comparison))

=== PERFORMANCE FROM 1935 ONWARDS ===
                                Annualized Return  Annualized Vol  \
Dynamic Rolling MSR(60 Months)           0.105989        0.123026   
Dynamic Rolling MSR(50 Months)           0.097706        0.125164   
Dynamic Rolling MSR(40 Months)           0.094981        0.124712   
Dynamic Rolling MSR(30 Months)           0.103250        0.133130   
Dynamic Rolling MSR(20 Months)           0.124286        0.143595   
Dynamic Rolling MSR(10 Months)           0.132326        0.147872   
Cap-weighted Market Index                0.109958        0.155409   

                                Sharpe Ratio  Skewness  Kurtosis  \
Dynamic Rolling MSR(60 Months)      0.601043 -0.096334  5.546790   
Dynamic Rolling MSR(50 Months)      0.526356 -0.306398  5.943905   
Dynamic Rolling MSR(40 Months)      0.506988 -0.449548  6.991691   
Dynamic Rolling MSR(30 Months)      0.535371 -0.298095  8.038369   
Dynamic Rolling MSR(20 Months)      0.638929 -0.269858  9.086377   
D

For this strategy, there also seems to be no particular trend as the length of the time window changes. This strategy implemented with all 6 time window lengths managed to score a higher sharpe ratio than the cap-weighted market index. The three time window lengths with the highest sharpe ratios, which happen to be the same as the Rolling MSR strategies are 60 months, 20 months, and 10 months, with sharpe ratios of 0.60, 0.64, and 0.67 respectively. On average, the Dynamic Rolling MSR strategy scored higher sharpe ratios than the Rolling MSR strategy. Compared to the market index, on average, this strategy has a much lower max drawdown, and slightly lower Cornish-Fisher VaR and Historic CVaR, indicating lower tail risk.

For this next strategy, I will allocate capital dynamically between the different portfolios(points) on the efficient frontier. This efficient frontier will be computed at the start of every month based on the returns during the previous 'n' months. The portfolios(points) on the efficient frontier contain the optimal weights that would have given you a particular return with the lowest risk(standard deviation) if you had implemented those weights across the past 'n' months. The efficient frontier contains portfolios with the highest returns for a given volatility or the lowest volatility for a given return. Therefore, there will naturally be portfolios with higher returns and higher risks, and ones with lower returns but also lower risks. This portfolio strategy will allocate more capital to higher return and higher risk portfolios when the previous month's return was positive, and more capital to lower return and lower risk portfolios when the previous month's return was negative.

Now I will develop a similar strategy to the Rolling MSR strategy, but instead of computing the optimal weights based on the maximum sharpe ratio, it will instead be on the highest return for a given level of risk(right end of the efficient frontier). This will(assuming no short selling) almost always mean, for a particular month, allocating 100% capital in the industry that had the highest average return during the past 'n' months. 

In [23]:
# ==================== ROLLING MAXIMUM RETURN (RIGHT END OF EF) ====================

def rolling_max_return_backtest(rets, window_months=60):
    """
    Rolling strategy that always picks the right-most point on the Efficient Frontier
    (i.e. the portfolio with the highest possible expected return).
    """
    portfolio_returns = []
    dates_used = []
    
    for t in range(window_months, len(rets)):
        # Training data = previous n months
        train = rets.iloc[t - window_months : t]
        
        # Expected returns (annualized)
        er = train.mean() * 12
        
        # Find the asset with the highest expected return
        best_asset = er.idxmax()
        
        # 100% allocation to that asset
        w = pd.Series(0.0, index=er.index)
        w[best_asset] = 1.0
        
        # Next month's actual return
        next_ret = rets.iloc[t] @ w
        
        portfolio_returns.append(next_ret)
        dates_used.append(rets.index[t])
    
    return pd.Series(portfolio_returns, index=dates_used, name="Rolling_Max_Return")

In [16]:
# ==================== RUN THE STRATEGY ====================

# Test a few different window lengths
windows = [60, 50, 40, 30, 20, 10]

results = {}
for w in windows:
    rets_series = rolling_max_return_backtest(ind_r, window_months=w)
    results[f"Rolling Max Return ({w}m)"] = rets_series['1935':]

comparison = pd.DataFrame(results)
comparison["MarketCap Weighted Index"] = market_rets

print("\n=== PERFORMANCE COMPARISON (from 1935) ===")
print(summary_stats(comparison))


=== PERFORMANCE COMPARISON (from 1935) ===
                          Annualized Return  Annualized Vol  Sharpe Ratio  \
Rolling Max Return (60m)           0.106027        0.284441      0.259783   
Rolling Max Return (50m)           0.100615        0.279327      0.245694   
Rolling Max Return (40m)           0.109315        0.275425      0.279924   
Rolling Max Return (30m)           0.127580        0.263184      0.360537   
Rolling Max Return (20m)           0.140064        0.263337      0.406466   
Rolling Max Return (10m)           0.168374        0.266161      0.505679   
MarketCap Weighted Index           0.109958        0.155409      0.500578   

                          Skewness  Kurtosis  Cornish-Fisher VaR (5%)  \
Rolling Max Return (60m)  0.312131  6.879542                 0.109362   
Rolling Max Return (50m)  0.206793  6.671541                 0.110532   
Rolling Max Return (40m)  0.025866  5.744882                 0.113875   
Rolling Max Return (30m)  0.070895  5.595138   

The results show us that this strategy performs very poorly in terms of the sharpe ratio and the max drawdown, even though the strategy implemented with some windows, specifically 20 months and 10 months, have high annualized returns. Comapred to the market index, this strategy has a much higher max drawdown, as well as higher Cornish-Fisher VaR and Historic CVaR, indicating a larger tail risk. 

Because we developed this strategy based on the highest return point on the efficient frontier, it is only fitting to also develop a strategy based on the point on the efficient frontier with the lowest risk, called the Global Minimum Variance (GMV) point. 

In [24]:
# ==================== ROLLING GMV BACKTEST ====================

def rolling_gmv_backtest(rets, window_months=60):
    """
    Rolling Global Minimum Variance strategy.
    """
    portfolio_returns = []
    dates_used = []
    
    for t in range(window_months, len(rets)):
        train = rets.iloc[t - window_months : t]
        cov = train.cov() * 12
        w = gmv(cov)                       # your course function
        next_ret = rets.iloc[t] @ w
        portfolio_returns.append(next_ret)
        dates_used.append(rets.index[t])
    
    return pd.Series(portfolio_returns, index=dates_used, name=f"GMV_{window_months}m")

def gmv(cov):
    """
    Returns the weights of the Global Minimum Volatility portfolio
    given a covariance matrix
    """
    n = cov.shape[0]
    return msr(0, np.repeat(1, n), cov)

In [22]:
# ==================== TEST MULTIPLE WINDOW LENGTHS ====================


# Choose the window lengths you want to test
windows = [60, 50, 40, 30, 20, 10]          # you can add/remove values here

results = {}

for w in windows:
    rets_series = rolling_gmv_backtest(ind_r, window_months=w)
    results[f"Rolling GMV ({w} months)"] = rets_series['1935':]


# Combine everything
comparison = pd.DataFrame(results)
comparison["MarketCap Weighted Index"] = market_rets

print("\n=== ROLLING GMV - MULTIPLE WINDOW COMPARISON (from 1935) ===")
print(summary_stats(comparison))


=== ROLLING GMV - MULTIPLE WINDOW COMPARISON (from 1935) ===
                          Annualized Return  Annualized Vol  Sharpe Ratio  \
Rolling GMV (60 months)            0.106597        0.120270      0.619742   
Rolling GMV (50 months)            0.109146        0.121692      0.632880   
Rolling GMV (40 months)            0.106533        0.122054      0.610170   
Rolling GMV (30 months)            0.103992        0.123770      0.581719   
Rolling GMV (20 months)            0.110354        0.126028      0.620431   
Rolling GMV (10 months)            0.113556        0.137786      0.590085   
MarketCap Weighted Index           0.109958        0.155409      0.500578   

                          Skewness   Kurtosis  Cornish-Fisher VaR (5%)  \
Rolling GMV (60 months)  -0.342244   6.092893                 0.049133   
Rolling GMV (50 months)  -0.370776   6.133091                 0.049856   
Rolling GMV (40 months)  -0.438625   6.030330                 0.050941   
Rolling GMV (30 months)  

The performance of this strategy is actually quite impressive compared to the other strategies as it achieves a consistent sharpe ratio centered around 0.6 for all windows. The lowest sharpe ratio is from the 30 month window strategy with 0.58 and the highest is from the 50 month window sgtrategy with 0.63. This strategy on average has a Higher sharpe ratio than the rolling MSR strategy. Although the highest sharpe ratio we saw from the Dynamic Rolling MSR strategy was 0.67 (from the 10 month window strategy), higher than 0.63, the sharpe ratios achieved by the Rolling GMV strategies is much more consistent across the different windows (centered around 0.6). Compared to the market index, this strategy, on average, has a moderately lower max drawdown. The Cornish-Fisher VaR and Historic CVaR of this strategy is also moderately lower, indicating reduced tail risk compared to the market index.

In [25]:
# ==================== DYNAMIC ROLLING GMV (INCREMENTAL) ====================

def dynamic_rolling_gmv(rets, 
                        window_months=60, 
                        riskfree_rate=0.03, 
                        sensitivity=8.0, 
                        max_gmv_weight=1.0):
    """
    Dynamic Rolling GMV with incremental allocation.
    """
    portfolio_returns = []
    gmv_weight_history = []
    dates_used = []
    
    current_gmv_weight = 1.0          # start fully invested
    
    for t in range(window_months, len(rets)):
        # Training window
        train = rets.iloc[t - window_months : t]
        cov = train.cov() * 12
        
        # Compute GMV portfolio
        w_gmv = gmv(cov)
        
        # This month's return
        this_month_gmv_ret = rets.iloc[t] @ w_gmv
        this_month_total_ret = (current_gmv_weight * this_month_gmv_ret) + \
                               ((1 - current_gmv_weight) * (riskfree_rate / 12))
        
        # Incremental update based on LAST month's return
        if t > window_months:
            last_month_ret = portfolio_returns[-1]
            current_gmv_weight = current_gmv_weight + sensitivity * last_month_ret
            current_gmv_weight = np.clip(current_gmv_weight, 0.0, max_gmv_weight)
        
        portfolio_returns.append(this_month_total_ret)
        gmv_weight_history.append(current_gmv_weight)
        dates_used.append(rets.index[t])
    
    result = pd.Series(portfolio_returns, index=dates_used, name="Dynamic_Rolling_GMV")
    weight_history = pd.Series(gmv_weight_history, index=dates_used, name="GMV_Weight")
    
    return result, weight_history          # ONLY 2 values returned

In [26]:
# ==================== TEST MULTIPLE WINDOW LENGTHS ====================

windows = [60, 50, 40, 30, 20, 10]

results = {}

for w in windows:
    dynamic_rets, _ = dynamic_rolling_gmv(ind_r, 
                                          window_months=w,
                                          sensitivity=8.0,
                                          max_gmv_weight=1.0)
    results[f"Dynamic GMV ({w} months)"] = dynamic_rets['1935':]

comparison = pd.DataFrame(results)
comparison["MarketCap Weighted Index"] = market_rets

print("\n=== DYNAMIC ROLLING GMV - MULTIPLE WINDOW COMPARISON ===")
print(summary_stats(comparison))


=== DYNAMIC ROLLING GMV - MULTIPLE WINDOW COMPARISON ===
                          Annualized Return  Annualized Vol  Sharpe Ratio  \
Dynamic GMV (60 months)            0.089398        0.084553      0.683648   
Dynamic GMV (50 months)            0.090308        0.085063      0.689967   
Dynamic GMV (40 months)            0.088491        0.084182      0.676172   
Dynamic GMV (30 months)            0.082747        0.085958      0.597152   
Dynamic GMV (20 months)            0.087095        0.087487      0.635092   
Dynamic GMV (10 months)            0.086646        0.094760      0.581712   
MarketCap Weighted Index           0.109958        0.155409      0.500578   

                          Skewness   Kurtosis  Cornish-Fisher VaR (5%)  \
Dynamic GMV (60 months)  -0.493471   9.358689                 0.032849   
Dynamic GMV (50 months)  -0.555942  10.415260                 0.032900   
Dynamic GMV (40 months)  -0.483672   9.080721                 0.032817   
Dynamic GMV (30 months)  -1.0

Although this strategy achieved a lower annualized return than the market index, it scores one of the highest sharpe ratios compared with the other strategies we have developed (0.69). The lowest sharpe ratio is from the 10 month window strategy with 0.58, and the highest is from the 50 month window with 0.69. Compared to the market index, this strategy has a much lower max drawdown. Compared to the market index, this strategy also has a much lower Cornish-Fisher VaR and Historic CVaR, indicating much lower tail risk.

I will now develop a strategy that allocates between the risk free asset, the Rolling GMV, and the Rolling MSR, where the allocation will depend on the previous month's return. It is similar to the Dynamic Rolling MSR and Dynamic Rolling GMV strategy, the only difference is I m allocating between three assets now, where the risk free asset is the safe asset, the Rolling MSR portfolio is the risky asset, and the Rolling GMV portfolio is the moderate risk asset.

In [27]:
# ==================== FIXED DYNAMIC THREE-ASSET STRATEGY ====================

def dynamic_three_asset(rets, 
                        window_months=60, 
                        riskfree_rate=0.03, 
                        sensitivity=6.0, 
                        start_date='1935'):
    """
    Dynamic allocation between:
      - Risk-free asset (safe)
      - Rolling GMV (moderate risk)
      - Rolling MSR (risky / high-Sharpe)
    """
    portfolio_returns = []
    weights_history = []          # [w_safe, w_gmv, w_msr]
    dates_used = []
    
    current_risk_level = 0.5      # 0 = all safe, 1 = all MSR
    
    for t in range(window_months, len(rets)):
        # Training window
        train = rets.iloc[t - window_months : t]
        er = train.mean() * 12
        cov = train.cov() * 12
        
        # Compute GMV and MSR with correct index
        w_gmv_np = gmv(cov)
        w_msr_np = msr(riskfree_rate, er, cov)
        
        # Convert to Series with correct asset names
        w_gmv = pd.Series(w_gmv_np, index=er.index)
        w_msr = pd.Series(w_msr_np, index=er.index)
        
        # Current allocation based on risk level
        if current_risk_level <= 0.5:
            w_safe = 1.0 - 2 * current_risk_level
            w_gmv_alloc = 2 * current_risk_level
            w_msr_alloc = 0.0
        else:
            w_safe = 0.0
            w_gmv_alloc = 2 * (1 - current_risk_level)
            w_msr_alloc = 2 * current_risk_level - 1.0
        
        # This month's portfolio return
        this_month_ret = (w_safe * (riskfree_rate / 12) +
                          w_gmv_alloc * (rets.iloc[t] @ w_gmv) +
                          w_msr_alloc * (rets.iloc[t] @ w_msr))
        
        # Incremental update of risk level
        if t > window_months:
            last_month_ret = portfolio_returns[-1]
            current_risk_level += sensitivity * last_month_ret
            current_risk_level = np.clip(current_risk_level, 0.0, 1.0)
        
        portfolio_returns.append(this_month_ret)
        weights_history.append([w_safe, w_gmv_alloc, w_msr_alloc])
        dates_used.append(rets.index[t])
    
    result = pd.Series(portfolio_returns, index=dates_used, name="Dynamic_3_Asset")
    weight_df = pd.DataFrame(weights_history, 
                             index=dates_used, 
                             columns=["Safe", "GMV", "MSR"])
    
    return result

In [28]:
# ==================== TEST MULTIPLE WINDOW LENGTHS ====================

windows = [60, 50, 40, 30, 20, 10]

results = {}

for w in windows:
    dynamic_rets = dynamic_three_asset(ind_r, window_months=w, sensitivity=6.0)
    results[f"Dynamic 3-Asset ({w} months)"] = dynamic_rets['1935':]

comparison = pd.DataFrame(results)
comparison["MarketCap Weighted Index"] = market_rets

print("\n=== DYNAMIC THREE-ASSET PERFORMANCE COMPARISON ===")
print(summary_stats(comparison))


=== DYNAMIC THREE-ASSET PERFORMANCE COMPARISON ===
                             Annualized Return  Annualized Vol  Sharpe Ratio  \
Dynamic 3-Asset (60 months)           0.106356        0.121884      0.609602   
Dynamic 3-Asset (50 months)           0.097704        0.119908      0.549414   
Dynamic 3-Asset (40 months)           0.090428        0.117433      0.500697   
Dynamic 3-Asset (30 months)           0.090570        0.124901      0.471844   
Dynamic 3-Asset (20 months)           0.104502        0.127210      0.569877   
Dynamic 3-Asset (10 months)           0.119431        0.132603      0.656280   
MarketCap Weighted Index              0.109958        0.155409      0.500578   

                             Skewness   Kurtosis  Cornish-Fisher VaR (5%)  \
Dynamic 3-Asset (60 months) -0.379156   6.263597                 0.050146   
Dynamic 3-Asset (50 months) -0.608995   7.242831                 0.051290   
Dynamic 3-Asset (40 months) -0.691040   6.498495                 0.051872   

This strategy, on average, achieves a lower sharpe ratio than the Dynamic Rolling GMV strategy. Compared to the market index, this strategy, on average, achieves a moderately lower max drawdown. The lower values of Corner-Fisher VaR and Historic CVaR also indicate lower tail risk compared to the market index.